# VocEd Lab 07 — End-to-End: N/C Ratio Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emilsar/VocEd/blob/main/07_nc_ratio_pipeline.ipynb)

This is the final lab.  Every notebook in this series — thresholding, optimisation, image
processing, k-NN, the minimal CNN, the U-Net — was building toward a single question:

> **Can an automated model predict the nucleus-to-cytoplasm (N/C) ratio accurately enough
> to be clinically useful?**

The N/C ratio is a key measurement in cytopathology.  A high ratio (large nucleus relative
to cytoplasm) is a hallmark of malignant cells.  A reliable automated pipeline would save
pathologists significant time and reduce inter-observer variability.

In this lab we:
1. Load the trained U-Net weights from Lab 06.
2. Run inference on the test images and compute predicted N/C ratios.
3. Compare predicted vs ground-truth N/C ratios.
4. Benchmark all methods from the series: hand-picked thresholds → optimised thresholds →
   U-Net.

By the end you will have a complete, data-driven answer to the clinical question.

## 0. Setup

In [ ]:
!pip install scikit-optimize --quiet

# Clone the repo
!git clone https://github.com/emilsar/VocEd.git
%cd VocEd

## 1. Load Data & Recreate the Train/Test Split

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from sklearn.model_selection import train_test_split

N = len(glob.glob('imagedata/X/*.npy'))
X = np.stack([np.load(f'imagedata/X/{i}.npy') for i in range(N)])
y = np.stack([np.load(f'imagedata/y/{i}.npy') for i in range(N)])

# Drop images with no nucleus pixels — identical to Lab 02
has_nucleus = (y == 2).sum(axis=(1, 2)) > 0
X, y = X[has_nucleus], y[has_nucleus]
N = len(X)

# Stratified split by nucleus size — identical to Lab 02
nucleus_size = (y == 2).sum(axis=(1, 2))
quartile     = np.digitize(nucleus_size, np.percentile(nucleus_size, [25, 50, 75]))

train_idx, test_idx = train_test_split(
    np.arange(N), test_size=0.2, stratify=quartile, random_state=42
)

mask_cmap = ListedColormap(['black', 'steelblue', 'crimson'])
legend_patches = [
    mpatches.Patch(color='black',     label='0 — background'),
    mpatches.Patch(color='steelblue', label='1 — cytoplasm'),
    mpatches.Patch(color='crimson',   label='2 — nucleus'),
]
print(f'{N} images retained  ({(~has_nucleus).sum()} with no nucleus removed)')
print(f'Test set: {len(test_idx)} images.')

## 2. The N/C Ratio Function

The nucleus-to-cytoplasm ratio is defined as:

$$\text{N/C ratio} = \frac{\text{number of nucleus pixels}}{\text{number of cytoplasm pixels}}$$

It is computed from the segmentation mask — so any method that produces a mask (thresholding,
k-NN, CNN, U-Net) can produce an N/C ratio estimate.

In [ ]:
def nc_ratio(mask):
    """
    Compute the N/C ratio from a segmentation mask.
    mask: (H, W) integer array with values 0 (bg), 1 (cyto), 2 (nucleus)
    Returns NaN if there are no cytoplasm pixels (avoids division by zero).
    """
    nucleus_pixels   = (mask == 2).sum()
    cytoplasm_pixels = (mask == 1).sum()
    if cytoplasm_pixels == 0:
        return float('nan')   # can't compute a ratio with no cytoplasm
    return nucleus_pixels / cytoplasm_pixels


# Ground-truth N/C ratios for all test images
gt_ratios = np.array([nc_ratio(y[i]) for i in test_idx])

# Remove any NaN images (very rare — only if cytoplasm class is completely absent)
valid = ~np.isnan(gt_ratios)
gt_ratios = gt_ratios[valid]
test_idx_valid = test_idx[valid]

print(f'Valid test images: {valid.sum()} / {len(test_idx)}')
print(f'Ground-truth N/C — mean: {gt_ratios.mean():.3f}  '
      f'std: {gt_ratios.std():.3f}  '
      f'range: [{gt_ratios.min():.3f}, {gt_ratios.max():.3f}]')

## 3. Method 1: Hand-Picked Thresholds (Lab 01)

In [ ]:
def segment_threshold(img, t_nucleus, t_background):
    gray = 0.299 * img[0] + 0.587 * img[1] + 0.114 * img[2]
    pred = np.zeros(gray.shape, dtype=np.int64)
    pred[gray < t_nucleus]                                = 2
    pred[(gray >= t_nucleus) & (gray < t_background)]    = 1
    return pred


# Lab 01 hand-picked values
ratios_lab01 = np.array([
    nc_ratio(segment_threshold(X[i], 0.45, 0.85))
    for i in test_idx_valid
])

mae_lab01 = np.nanmean(np.abs(ratios_lab01 - gt_ratios))
print(f'Lab 01 hand-picked thresholds — MAE: {mae_lab01:.4f}')

## 4. Method 2: Bayesian-Optimised Thresholds (Lab 02)

In [ ]:
from skopt import gp_minimize
from skopt.space import Real

def dice_score_np(pred, target, cls):
    pred_mask   = (pred   == cls)
    target_mask = (target == cls)
    intersection = (pred_mask & target_mask).sum()
    denom = pred_mask.sum() + target_mask.sum()
    return 1.0 if denom == 0 else 2 * intersection / denom

def objective_lab02(params):
    t_nuc, t_bg = params
    scores = []
    for i in train_idx:
        pred = segment_threshold(X[i], t_nuc, t_bg)
        d = (dice_score_np(pred, y[i], 1) + dice_score_np(pred, y[i], 2)) / 2
        scores.append(d)
    return -np.mean(scores)

print('Re-running Lab 02 Bayesian optimisation...')
res02 = gp_minimize(objective_lab02,
                    [Real(0.10, 0.70), Real(0.50, 0.99)],
                    n_calls=50, n_initial_points=10, random_state=42, verbose=False)
best_nuc, best_bg = res02.x
print(f'Optimised thresholds: t_nuc={best_nuc:.4f}  t_bg={best_bg:.4f}')

ratios_lab02 = np.array([
    nc_ratio(segment_threshold(X[i], best_nuc, best_bg))
    for i in test_idx_valid
])
mae_lab02 = np.nanmean(np.abs(ratios_lab02 - gt_ratios))
print(f'Lab 02 optimised thresholds — MAE: {mae_lab02:.4f}')

## 5. Method 3: U-Net (Lab 06)

We re-define the same `SmallUNet` architecture and load the weights saved at the end
of Lab 06.  If you did not run Lab 06, execute it first and make sure `unet_weights.pt`
is present in the working directory.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class SmallUNet(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.enc1      = ConvBlock(3,   32)
        self.enc2      = ConvBlock(32,  64)
        self.pool      = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(64, 128)
        self.dec2      = ConvBlock(128 + 64, 64)
        self.dec1      = ConvBlock(64  + 32, 32)
        self.out_conv  = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(self.pool(s1))
        b  = self.bottleneck(self.pool(s2))
        u2 = F.interpolate(b,  scale_factor=2, mode='bilinear', align_corners=False)
        u2 = self.dec2(torch.cat([u2, s2], dim=1))
        u1 = F.interpolate(u2, scale_factor=2, mode='bilinear', align_corners=False)
        u1 = self.dec1(torch.cat([u1, s1], dim=1))
        return self.out_conv(u1)


# Load the saved weights
unet = SmallUNet(num_classes=3).to(device)
# map_location ensures weights load correctly even if the model was trained on GPU
# but this notebook is running on CPU (or vice versa)
unet.load_state_dict(torch.load('unet_weights.pt', map_location=device))
unet.eval()   # inference mode — disables BatchNorm updates and dropout
print('U-Net weights loaded.')

In [ ]:
# ── Run U-Net inference on all valid test images ──────────────────────────────
ratios_unet = []

with torch.no_grad():
    for i in test_idx_valid:
        img_t  = torch.from_numpy(X[i]).unsqueeze(0).to(device)  # (1, 3, 256, 256)
        logits = unet(img_t)                                       # (1, 3, 256, 256)
        pred   = logits.argmax(dim=1).squeeze().cpu().numpy()      # (256, 256)
        ratios_unet.append(nc_ratio(pred))

ratios_unet = np.array(ratios_unet)
mae_unet = np.nanmean(np.abs(ratios_unet - gt_ratios))
print(f'U-Net — MAE: {mae_unet:.4f}')

## 6. Scatter Plots: Predicted vs Ground-Truth N/C Ratio

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

methods = [
    ('Lab 01 — hand-picked', ratios_lab01, mae_lab01, 'steelblue'),
    ('Lab 02 — optimised',   ratios_lab02, mae_lab02, 'darkorange'),
    ('Lab 06 — U-Net',       ratios_unet,  mae_unet,  'crimson'),
]

max_ratio = max(gt_ratios.max(),
                np.nanmax(ratios_lab01),
                np.nanmax(ratios_lab02),
                np.nanmax(ratios_unet)) * 1.1

for ax, (title, pred_ratios, mae, color) in zip(axes, methods):
    ax.scatter(gt_ratios, pred_ratios, alpha=0.6, color=color, s=30)
    # Perfect prediction line: if predicted = ground truth, points lie on this line
    ax.plot([0, max_ratio], [0, max_ratio], 'k--', linewidth=1, label='Perfect prediction')
    ax.set_xlabel('Ground-truth N/C ratio')
    ax.set_ylabel('Predicted N/C ratio')
    ax.set_title(f'{title}\nMAE = {mae:.4f}')
    ax.set_xlim(0, max_ratio)
    ax.set_ylim(0, max_ratio)
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

print('Points near the dashed line = accurate predictions.')
print('Points far above/below = systematic over/under-estimation.')

## 7. Final Benchmark Summary

In [ ]:
# ── Per-image error table (first 10 test images) ──────────────────────────────
print(f'{'Image':>6}  {'GT ratio':>10}  {'Lab01':>8}  {'Lab02':>8}  {'U-Net':>8}')
print('-' * 52)
for k in range(min(10, len(test_idx_valid))):
    i = test_idx_valid[k]
    print(f'{i:6d}  {gt_ratios[k]:10.3f}  '
          f'{ratios_lab01[k]:8.3f}  '
          f'{ratios_lab02[k]:8.3f}  '
          f'{ratios_unet[k]:8.3f}')

print('\n' + '=' * 60)
print(f'{"Method":<40}  {"N/C MAE":>8}')
print('-' * 60)
print(f'{"Lab 01 — hand-picked thresholds":<40}  {mae_lab01:8.4f}')
print(f'{"Lab 02 — Bayesian optimised thresholds":<40}  {mae_lab02:8.4f}')
print(f'{"Lab 06 — Small U-Net":<40}  {mae_unet:8.4f}')
print('=' * 60)

In [ ]:
# ── Error distribution plot ───────────────────────────────────────────────────
errors_lab01 = np.abs(ratios_lab01 - gt_ratios)
errors_lab02 = np.abs(ratios_lab02 - gt_ratios)
errors_unet  = np.abs(ratios_unet  - gt_ratios)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(errors_lab01, bins=15, alpha=0.5, color='steelblue',  label=f'Lab01  MAE={mae_lab01:.3f}')
ax.hist(errors_lab02, bins=15, alpha=0.5, color='darkorange', label=f'Lab02  MAE={mae_lab02:.3f}')
ax.hist(errors_unet,  bins=15, alpha=0.5, color='crimson',    label=f'U-Net  MAE={mae_unet:.3f}')
ax.set_xlabel('Absolute N/C ratio error')
ax.set_ylabel('Number of images')
ax.set_title('Distribution of N/C Ratio Errors (test set)')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Qualitative Inspection of Best and Worst Cases

In [ ]:
# ── Show U-Net's best and worst N/C ratio predictions ────────────────────────
order   = np.argsort(errors_unet)   # sort by ascending error
best4   = order[:4]    # smallest errors
worst4  = order[-4:]   # largest errors

def show_grid(rel_indices, title):
    fig, axes = plt.subplots(len(rel_indices), 3, figsize=(12, 4 * len(rel_indices)))
    if len(rel_indices) == 1:
        axes = axes[np.newaxis, :]   # ensure axes is 2-D even for 1 row
    for row, ri in enumerate(rel_indices):
        abs_i = test_idx_valid[ri]
        img_t = torch.from_numpy(X[abs_i]).unsqueeze(0).to(device)
        with torch.no_grad():
            pred = unet(img_t).argmax(dim=1).squeeze().cpu().numpy()

        axes[row, 0].imshow(X[abs_i].transpose(1, 2, 0))
        axes[row, 0].set_title(f'Image {abs_i} — RGB');  axes[row, 0].axis('off')

        axes[row, 1].imshow(y[abs_i], cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest')
        axes[row, 1].set_title(f'GT mask  N/C={gt_ratios[ri]:.3f}');  axes[row, 1].axis('off')

        axes[row, 2].imshow(pred, cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest')
        axes[row, 2].set_title(f'Predicted  N/C={ratios_unet[ri]:.3f}  err={errors_unet[ri]:.3f}')
        axes[row, 2].axis('off')

    fig.suptitle(title, fontsize=13)
    fig.legend(handles=legend_patches, loc='lower center', ncol=3, fontsize=10,
               bbox_to_anchor=(0.5, 0.0))
    plt.tight_layout(rect=[0, 0.06, 1, 0.96])
    plt.show()

show_grid(best4,  'U-Net — Best N/C Ratio Predictions')
show_grid(worst4, 'U-Net — Worst N/C Ratio Predictions')

## Wrap-up — All 7 Labs

**What we built, one step at a time:**

```
Lab 01  hand-picked thresholds      →  baseline: grayscale rule works, but fragile
Lab 02  Bayesian-optimised thresh.  →  automatic search beats hand-picking
Lab 03  image processing pipeline   →  classical pre/post-processing helps at the margins
Lab 04  k-NN on pixel colours       →  full RGB better than grayscale; no spatial context
Lab 05  minimal CNN                 →  learns spatial filters; still no global view
Lab 06  small U-Net                 →  multi-scale; best Dice in the series
Lab 07  N/C ratio pipeline          →  answers the clinical question
```

**Key lesson:** Each lab's method improvement translated — to varying degrees — into a
better N/C ratio estimate.  Deep learning (U-Net) achieved the lowest MAE, but the
improvement over optimised thresholds may be smaller than expected.  This is common in
structured medical imaging: classical methods can be surprisingly competitive when the
data has clear structure.

---

## Group Exercise — Clinical Threshold

A pathologist tells you that she needs the N/C ratio estimate to be within **±0.05** of
the ground truth to make a reliable diagnosis.

```python
# What fraction of test images are within ±0.05?
within_lab01 = (np.abs(ratios_lab01 - gt_ratios) <= 0.05).mean()
within_unet  = (np.abs(ratios_unet  - gt_ratios) <= 0.05).mean()
print(f'Lab01 within ±0.05: {within_lab01:.1%}')
print(f'U-Net within ±0.05: {within_unet:.1%}')
```

**Person A** — Compute the above for all three methods (Lab01, Lab02, U-Net).

**Person B** — Look at the worst-performing U-Net images (highest error).  What do those
cells look like?  Do they share common visual features (unusual staining, multiple cells,
debris)?

**Person C** — Imagine you wanted to improve the U-Net further.  Based on what you saw
in the failure cases, what one change would you try first?  (Data augmentation?  More
training data?  A deeper model?  Post-processing?)

Share your answers and discuss as a group.